# Python Tutorial: Analyze a Social Network

This notebook follows the Ladybug Python tutorial. It creates a Ladybug database from a small social network dataset, imports CSV files, and runs Cypher queries against the graph.

## Setup

Install Ladybug, download the tutorial dataset, and unzip it into the notebook working directory.

In [1]:
%pip install ladybug

/Users/arun/src/ladybug-icebug-notebook/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

DATA_URL = "https://huggingface.co/datasets/ladybugdb/python-tutorial/resolve/main/tutorial_data.zip"
zip_path = Path("tutorial_data.zip")
data_dir = Path("tutorial_data")

if not data_dir.exists():
    urlretrieve(DATA_URL, zip_path)
    with ZipFile(zip_path) as archive:
        archive.extractall(".")
    zip_path.unlink()

sorted(str(path) for path in data_dir.rglob("*.csv"))

['tutorial_data/node/post.csv',
 'tutorial_data/node/user.csv',
 'tutorial_data/relation/FOLLOWS.csv',
 'tutorial_data/relation/LIKES.csv',
 'tutorial_data/relation/POSTS.csv']

## Create the database

Create a new on-disk Ladybug database named `social_network.lbug`. The cell removes an existing tutorial database first, so it can be rerun without primary-key conflicts.

In [3]:
import shutil

import ladybug as lb

db_path = Path("social_network.lbug")
if db_path.is_dir():
    shutil.rmtree(db_path)
elif db_path.exists():
    db_path.unlink()

db = lb.Database(str(db_path))
conn = lb.Connection(db)

### Define schema

The dataset has `User` and `Post` node tables, plus `FOLLOWS`, `POSTS`, and `LIKES` relationship tables.

In [4]:
conn.execute("""
    CREATE NODE TABLE User (
        user_id INT64 PRIMARY KEY,
        username STRING,
        account_creation_date DATE
    )""")
conn.execute("""
    CREATE NODE TABLE Post (
        post_id INT64 PRIMARY KEY,
        post_date DATE,
        like_count INT64,
        retweet_count INT64
    )""")
conn.execute("""
    CREATE REL TABLE FOLLOWS (
        FROM User TO User
    )""")
conn.execute("""
    CREATE REL TABLE POSTS (
        FROM User TO Post
    )""")
conn.execute("""
    CREATE REL TABLE LIKES (
        FROM User TO Post
    )""")

### Import CSV data

In [5]:
conn.execute("COPY User FROM './tutorial_data/node/user.csv'")
conn.execute("COPY Post FROM './tutorial_data/node/post.csv'")
conn.execute("COPY FOLLOWS FROM './tutorial_data/relation/FOLLOWS.csv'")
conn.execute("COPY POSTS FROM './tutorial_data/relation/POSTS.csv'")
conn.execute("COPY LIKES FROM './tutorial_data/relation/LIKES.csv'")

### Show table information

In [6]:
for row in conn.execute("CALL SHOW_TABLES() RETURN *"):
    print(row)

[0, 'User', 'NODE', 'main(graph)', '']
[1, 'Post', 'NODE', 'main(graph)', '']
[3, 'FOLLOWS', 'REL', 'main(graph)', '']
[5, 'POSTS', 'REL', 'main(graph)', '']
[7, 'LIKES', 'REL', 'main(graph)', '']


## Query the graph

The helper below runs a Cypher query and prints each row returned by Ladybug.

In [7]:
def run(query: str, params: dict | None = None) -> None:
    result = conn.execute(query, params) if params else conn.execute(query)
    for row in result:
        print(row)

### Q1: Which user has the most followers? And how many followers do they have?

First, list a few users who are followed by another user.

In [8]:
run("""
MATCH (u1:User)-[f:FOLLOWS]->(u2:User)
RETURN u2.username
LIMIT 5
""")

['coolwolf752']
['stormfox762']
['stormninja678']
['darkdog878']
['brightninja683']


Count how many times each user appears as a followee.

In [9]:
run("""
MATCH (u1:User)-[f:FOLLOWS]->(u2:User)
RETURN u2.username, COUNT(u2) AS follower_count
LIMIT 5
""")

['mysticwolf198', 4]
['stormcat597', 2]
['coolking201', 3]
['epiccat105', 4]
['brightninja683', 5]


Order users by follower count and return one of the most-followed users.

In [10]:
run("""
MATCH (u1:User)-[f:FOLLOWS]->(u2:User)
RETURN u2.username, COUNT(u2) AS follower_count
ORDER BY follower_count DESC
LIMIT 1
""")

['darkdog878', 6]


The dataset has multiple users tied for the greatest follower count. This query returns every user with the maximum count.

In [11]:
run("""
MATCH (u1:User)-[f:FOLLOWS]->(u2:User)
WITH u2, COUNT(u1) as follower_count
WITH MAX(follower_count) as max_count
MATCH (u1:User)-[f:FOLLOWS]->(u2:User)
WITH u2, COUNT(u1) as follower_count, max_count
WHERE follower_count = max_count
RETURN u2.username, follower_count
""")

['stormninja678', 6]
['darkdog878', 6]


Expected result:

```text
['stormninja678', 6]
['darkdog878', 6]
```

### Q2: What is the shortest path between two users?

Find the shortest `FOLLOWS` path from `silentguy245` to `epicwolf202`.

In [12]:
run("""
MATCH p = (u1:User)-[f:FOLLOWS* SHORTEST]->(u2:User)
WHERE u1.username = 'silentguy245'
AND u2.username = 'epicwolf202'
RETURN p
""")

[{'_NODES': [{'_ID': {'offset': 19, 'table': 0}, '_LABEL': 'User', 'user_id': 20, 'username': 'silentguy245', 'account_creation_date': datetime.date(2022, 10, 11)}, {'_ID': {'offset': 14, 'table': 0}, '_LABEL': 'User', 'user_id': 15, 'username': 'mysticwolf198', 'account_creation_date': datetime.date(2021, 1, 4)}, {'_ID': {'offset': 0, 'table': 0}, '_LABEL': 'User', 'user_id': 1, 'username': 'epicwolf202', 'account_creation_date': datetime.date(2022, 9, 9)}], '_RELS': [{'_SRC': {'offset': 19, 'table': 0}, '_DST': {'offset': 14, 'table': 0}, '_LABEL': 'FOLLOWS', '_ID': {'offset': 64, 'table': 2}}, {'_SRC': {'offset': 14, 'table': 0}, '_DST': {'offset': 0, 'table': 0}, '_LABEL': 'FOLLOWS', '_ID': {'offset': 47, 'table': 2}}]}]


The raw recursive relationship value is useful, but verbose. Return only the usernames in the path to make the result easier to read.

In [13]:
run("""
MATCH p = (u1:User)-[f:FOLLOWS* SHORTEST]->(u2:User)
WHERE u1.username = 'silentguy245'
AND u2.username = 'epicwolf202'
RETURN PROPERTIES(NODES(p), 'username')
""")

[['silentguy245', 'mysticwolf198', 'epicwolf202']]


Expected result:

```text
[['silentguy245', 'mysticwolf198', 'epicwolf202']]
```

### Q3: How many 3-hop paths exist between userA and userB that pass through userC?

First, count all paths of length 3 in the graph.

In [14]:
run("""
MATCH (u1:User)-[f1:FOLLOWS]->(u2:User)-[f2:FOLLOWS]->(u3:User)-[f3:FOLLOWS]->(u4:User)
RETURN COUNT(u4)
""")

[667]


Use a prepared statement to count paths from `epicwolf202` to `stormcat597` that pass through `stormfox762`.

In [15]:
query = """
    MATCH (u1:User)-[f1:FOLLOWS]->(u2:User)-[f2:FOLLOWS]->(u3:User)-[f3:FOLLOWS]->(u4:User)
    WHERE u1.username = $usersrc
    AND u4.username = $userdst
    AND (u2.username = $userint OR u3.username = $userint)
    RETURN COUNT(u4)
    """
prepared_stmt = conn.prepare(query)
params = {
    "usersrc": "epicwolf202",
    "userdst": "stormcat597",
    "userint": "stormfox762",
}

for row in conn.execute(prepared_stmt, params):
    print(row)

[1]


/var/folders/mg/8cz953zd56g5kn6ls6xpy7t80000gn/T/ipykernel_43077/689780432.py:8: DeprecationWarning: The use of separate prepare + execute of queries is deprecated. Please using a single call to the execute() API instead.
  prepared_stmt = conn.prepare(query)


Expected result:

```text
['COUNT(u4._ID)']
[1]
```

You are now ready to import your own datasets into Ladybug and query them using Ladybug's Python API.